In [1]:
from pathlib import Path
import sys
import os
import json
import math
import datetime
import warnings
from datetime import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import joblib
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
)
import findspark
findspark.init()

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as func
from pyspark.sql.functions import col, desc, udf
from pyspark.sql.types import StructType, StructField, StringType

from ortools.linear_solver import pywraplp

warnings.filterwarnings("ignore")


# Backflow imports
from backflow import exp_setting
from backflow.programming import pgt
from backflow.calib import metrics as calib_metric
from backflow.calib import stats as calib_stats
from backflow.visualization import vis
from backflow.programming import simulation as pgtsimu
from backflow.utils import tools as bkfl_tools


# Matplotlib font

from pyspark.sql import SparkSession
spark3conf_best = {
    "spark.submit.deployMode": "client",
    "spark.yarn.queue": "root.growstrategy_group_prod",

    "spark.driver.memory": "24g",
    "spark.driver.memoryOverhead": "4g",

    "spark.dynamicAllocation.enabled": "true",
    "spark.dynamicAllocation.minExecutors": "80",
    "spark.dynamicAllocation.initialExecutors": "80",
    "spark.dynamicAllocation.maxExecutors": "120",
    "spark.dynamicAllocation.shuffleTracking.enabled": "true",

    "spark.executor.cores": "2",
    "spark.executor.memory": "12g",
    "spark.executor.memoryOverhead": "8g",

    "spark.default.parallelism": "1200",
    "spark.sql.shuffle.partitions": "1200",

    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "134217728",

    "spark.sql.autoBroadcastJoinThreshold": "104857600",
    "spark.sql.broadcastTimeout": "3000",

    "spark.network.timeout": "600s",
    "spark.executor.heartbeatInterval": "60s",

    "spark.memory.fraction": "0.8",
    "spark.memory.storageFraction": "0.3",

    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.execution.arrow.maxRecordsPerBatch": "5000",

    "mapred.input.dir.recursive": "TRUE",
    "hive.mapred.supports.subdirectories": "TRUE",

    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryoserializer.buffer.max": "1024m",

    "spark.yarn.dist.archives": "hdfs://difed/user/dm/ENV/dlflow/v2/py3.7.15_for_tfcluster_63.tgz",
    "spark.executorEnv.PYSPARK_PYTHON": "./py3.7.15_for_tfcluster_63.tgz/py3.7.15_for_tfcluster/bin/python3",

    "spark.extraListeners": "sparkmonitor.listener.JupyterSparkMonitorListener",

    "spark.sql.merge.mode": "FAST",
    "spark.sql.merge.enabled": "TRUE",
}

builder = (
    SparkSession
    .builder
    .appName("flowback_euen_v22")
    .master("yarn")
)

for k, v in spark3conf_best.items():
    builder = builder.config(k, v)

spark = builder.enableHiveSupport().getOrCreate()

print("[Info] Spark Version:", spark.version)
print("[Info] Spark Queue:", spark.conf.get("spark.yarn.queue"))
print("[Info] Spark App ID:", spark.sparkContext.applicationId)
print(spark.conf.get("spark.yarn.queue"))
print(spark.sparkContext.applicationId)

load metric func from causalml...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Application Id: application_1782979140233_76418696, Tracking URL: http://bigdata-nmg-hdprm00.nmg01:8088/proxy/application_1782979140233_76418696/
INFO:SparkMonitorKernel:Client Connected ('127.0.0.1', 51902)


[Info] Spark Version: 3.2.0-101
[Info] Spark Queue: root.growstrategy_group_prod
[Info] Spark App ID: application_1782979140233_76418696
root.growstrategy_group_prod
application_1782979140233_76418696


In [2]:
# 查看euen训练样本

audit_detail_df = spark.sql("""
with feature_data as (
    select
        flowtree_group_name,
        group_name_eve,
        dt,
        pid
    from dm.ut_alg_flowback_msg_outcall_feature_train_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

label_data as (
    select
        flowtree_group_name,
        group_name_eve,
        dt,
        pid,
        finish_cnt_7d
    from dm.ut_alg_flowback_msg_outcall_label_v2_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

joined_data as (
    select
        f.pid,
        f.dt,
        f.flowtree_group_name,
        f.group_name_eve,
        l.finish_cnt_7d,
        sum(l.finish_cnt_7d) over (
            partition by f.pid
        ) as finish_all_cnt
    from feature_data f
    join label_data l
      on f.pid = l.pid
     and f.dt = l.dt
     and f.flowtree_group_name = l.flowtree_group_name
     and f.group_name_eve = l.group_name_eve
),

-- 完全复现旧 SQL 最后的 outcome 筛选
old_train_rows as (
    select *
    from joined_data
    where
        (finish_all_cnt > 0 and finish_cnt_7d > 0)
        or finish_all_cnt = 0
),

-- 检查同一个 pid+dt+组在最终数据中是否仍有多行
key_stats as (
    select
        pid,
        dt,
        group_name_eve,
        count(*) as key_row_cnt,
        count(distinct finish_cnt_7d) as label_value_cnt,
        max(finish_cnt_7d) as finish_cnt_7d
    from old_train_rows
    group by pid, dt, group_name_eve
),

ordered_data as (
    select
        *,
        lag(dt) over (
            partition by pid, group_name_eve
            order by dt
        ) as previous_dt,
        lag(finish_cnt_7d) over (
            partition by pid, group_name_eve
            order by dt
        ) as previous_finish_cnt_7d
    from key_stats
)

select
    *,
    datediff(to_date(dt), to_date(previous_dt)) as gap_days,
    case
        when previous_dt is not null
         and datediff(to_date(dt), to_date(previous_dt)) between 1 and 6
            then 1
        else 0
    end as repeat_7d_flag
from ordered_data
""")

audit_detail_df.createOrReplaceTempView(
    "v5_euen_train_audit_detail"
)

In [3]:
multi_positive_pdf = spark.sql("""
select
    count(*) as repeat_pair_cnt,

    sum(
        case
            when previous_finish_cnt_7d > 0
             and finish_cnt_7d > 0
            then 1 else 0
        end
    ) as multi_positive_pair_cnt,

    count(distinct
        case
            when previous_finish_cnt_7d > 0
             and finish_cnt_7d > 0
            then pid
        end
    ) as multi_positive_user_cnt,

    sum(
        case
            when previous_finish_cnt_7d > 0
             and finish_cnt_7d > 0
             and previous_finish_cnt_7d = finish_cnt_7d
            then 1 else 0
        end
    ) as same_value_positive_pair_cnt,

    sum(
        case
            when previous_finish_cnt_7d > 0
             and finish_cnt_7d > 0
             and previous_finish_cnt_7d <> finish_cnt_7d
            then 1 else 0
        end
    ) as different_value_positive_pair_cnt

from v5_euen_train_audit_detail
where repeat_7d_flag = 1
""").toPandas()

multi_positive_pdf

,repeat_pair_cnt,multi_positive_pair_cnt,multi_positive_user_cnt,same_value_positive_pair_cnt,different_value_positive_pair_cnt
0,7147126,2742,2735,704,2038
